# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labanaprince72-a11y/internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds a transparent content-refresh baseline on the anonymized starter slice. The score is decision support, not a claim that every ranked page needs an edit. I keep the retrospective decline outcome separate from the score so it can be used only for an honest after-the-fact check.

The notebook uses the local file `data/raw/content_refresh_anonymized.csv`, which is reproducible in the repository and contains pseudonymized IDs only.


## 1. My rule and its reason codes

**Lane:** content refresh and search-action prioritization.

**Plain-words rule:** first prioritize pages with enough search visibility to matter. Add points when a visible page sits on page 1–2 but has CTR at or below 1%, because the snippet or intent match may be underperforming. Add more points when the page has not been updated for at least 180 days, because a stale page is a reasonable refresh candidate. Rank by this fixed score; do not fit weights to the outcome.

**Score:** visibility (0–2) + position/CTR opportunity (0–2) + staleness (0–2).

**Reason codes and actions:**

- `stale_visible_low_ctr` → `refresh_and_recheck`: visible, stale, and low-CTR opportunity.
- `stale_visible` → `refresh_and_recheck`: visible and stale, but no qualifying position/CTR signal.
- `visible_page1_2_low_ctr` → `improve_snippet_or_intent`: visible page-1/2 opportunity with CTR ≤ 1%.
- `visible_page1_2` → `review_search_fit`: visible page-1/2 page without the low-CTR trigger.
- `visible_only` → `monitor_or_research`: visible but without the stronger refresh signals.
- `low_visibility` → `monitor`: below the 300-impression visibility floor.

The two signal checks below are intentionally descriptive. A retrospective decline flag is printed only after the score is built; it is never an input to the score.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected {DATA_PATH}; run this notebook from the repository root.")

df = pd.read_csv(DATA_PATH)
required = {
    "content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "days_since_last_update", "trend_direction"
}
missing = sorted(required - set(df.columns))
assert not missing, f"Missing required columns: {missing}"

# This is an outcome-only audit field. It is never included in score_inputs.
df["observed_decline_outcome"] = (df["trend_direction"] == "down").astype(int)

# Signal check 1: visibility, linked to the session's volume / quick-win logic.
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 299, 2_999, np.inf],
    labels=["low (<300)", "moderate (300-2,999)", "high (3,000+)"]
).astype(str)
volume_check = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_clicks=("clicks_90d", "median"),
          median_ctr_pct=("ctr", "median")
      )
      .reset_index()
)
print(f"Rows loaded: {len(df):,}")
print(f"Retrospective decline base rate (audit only): {df['observed_decline_outcome'].mean():.1%}")
print("\nSignal 1 — visibility buckets (volume-linked):")
print(volume_check.to_string(index=False))
print("Verdict: CONFIRMED — higher-visibility buckets have materially more median clicks, so visibility is relevant to prioritization.")

# Signal check 2: a position/CTR opportunity, linked to the session's CTR-vs-position logic.
df["position_ctr_signal"] = "other"
df.loc[df["avg_position"].eq(0), "position_ctr_signal"] = "no position data"
df.loc[
    (df["impressions_90d"] >= 300)
    & df["avg_position"].between(1, 10)
    & (df["ctr"] <= 1.0),
    "position_ctr_signal"
] = "visible page 1 + low CTR"
df.loc[
    (df["impressions_90d"] >= 300)
    & df["avg_position"].gt(10)
    & df["avg_position"].le(20)
    & (df["ctr"] <= 1.0),
    "position_ctr_signal"
] = "visible page 2 + low CTR"
position_ctr_check = (
    df.groupby("position_ctr_signal", sort=False, observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_ctr_pct=("ctr", "median"),
          median_position=("avg_position", "median")
      )
      .reset_index()
)
print("\nSignal 2 — position/CTR opportunity buckets:")
print(position_ctr_check.to_string(index=False))
print("Verdict: MIXED — the visible page-1/2 low-CTR groups are actionable, but low CTR is common in this slice, so it is not proof that a change will work.")

score_inputs = ["impressions_90d", "avg_position", "ctr", "days_since_last_update"]
forbidden_inputs = {"trend_direction", "trend_pct", "observed_decline_outcome", "is_declining_label"}
assert not forbidden_inputs.intersection(score_inputs)


Rows loaded: 30,000
Retrospective decline base rate (audit only): 54.2%

Signal 1 — visibility buckets (volume-linked):
       volume_bucket     n  median_impressions  median_clicks  median_ctr_pct
       high (3,000+)  8283              8426.0           20.0            0.21
          low (<300) 11248                31.0            0.0            0.00
moderate (300-2,999) 10469               998.0            1.0            0.12
Verdict: CONFIRMED — higher-visibility buckets have materially more median clicks, so visibility is relevant to prioritization.

Signal 2 — position/CTR opportunity buckets:
     position_ctr_signal     n  median_impressions  median_ctr_pct  median_position
visible page 2 + low CTR  4880              1754.0            0.16             14.1
                   other 16186               164.0            0.00             21.0
visible page 1 + low CTR  7729              3899.0            0.22              6.4
        no position data  1205                 1.0        

## 2. Build the ranked queue (writes the CSV)

The thresholds are fixed before looking at the ranked outcomes: 300 impressions is the minimum visibility floor, 3,000 impressions is the high-visibility tier, page 1–2 means average position from 1 through 20, CTR ≤ 1% is the low-CTR trigger, and 180 days is the stale-content trigger.

The exported queue contains pseudonymous IDs and pre-decision fields only. The observed decline outcome is used in a separate retrospective metric and is not exported as an input to the action rule.


In [2]:
# Transparent, hand-written score: no fitted weights and no label-derived inputs.
visible = df["impressions_90d"] >= 300
high_visibility = df["impressions_90d"] >= 3_000
position_1_to_20 = df["avg_position"].between(1, 20)
low_ctr = df["ctr"] <= 1.0
stale = df["days_since_last_update"] >= 180

visibility_score = np.select([high_visibility, visible], [2, 1], default=0)
opportunity_score = np.select(
    [visible & position_1_to_20 & low_ctr, visible & position_1_to_20],
    [2, 1],
    default=0
)
staleness_score = np.select(
    [stale, df["days_since_last_update"] >= 90],
    [2, 1],
    default=0
)
df["score"] = visibility_score + opportunity_score + staleness_score

def reason_code(row):
    if row["stale"] and row["visible"] and row["low_ctr_opportunity"]:
        return "stale_visible_low_ctr"
    if row["stale"] and row["visible"]:
        return "stale_visible"
    if row["visible"] and row["position_1_to_20"] and row["low_ctr"]:
        return "visible_page1_2_low_ctr"
    if row["visible"] and row["position_1_to_20"]:
        return "visible_page1_2"
    if row["visible"]:
        return "visible_only"
    return "low_visibility"

def action_for(code):
    return {
        "stale_visible_low_ctr": "refresh_and_recheck",
        "stale_visible": "refresh_and_recheck",
        "visible_page1_2_low_ctr": "improve_snippet_or_intent",
        "visible_page1_2": "review_search_fit",
        "visible_only": "monitor_or_research",
        "low_visibility": "monitor",
    }[code]

def confidence_for(row):
    if row["reason_code"] == "stale_visible_low_ctr":
        return "medium-high: two independent review signals, but outcome still needs human confirmation"
    if row["reason_code"] in {"stale_visible", "visible_page1_2_low_ctr"}:
        return "medium: one strong signal plus useful visibility"
    if row["reason_code"] == "visible_page1_2":
        return "medium-low: rank opportunity is visible but CTR is not the trigger"
    return "low: weak prioritization evidence"

def what_would_make_it_wrong(row):
    if row["reason_code"].startswith("stale"):
        return "The update date may be missing or the page may already be scheduled for a refresh."
    if "low_ctr" in row["reason_code"]:
        return "The 90-day click count may be too small, or the query mix may not match the page intent."
    if row["reason_code"] == "visible_page1_2":
        return "Average position may be unstable across queries, so the page may not have one clear ranking problem."
    if row["reason_code"] == "visible_only":
        return "Traffic may be branded, seasonal, or already healthy enough that editing would add risk."
    return "Low volume may make the observed signal too noisy to justify work."

# Helper flags are rule inputs, not outcome fields.
df["visible"] = visible
df["stale"] = stale
df["position_1_to_20"] = position_1_to_20
df["low_ctr"] = low_ctr
df["low_ctr_opportunity"] = visible & position_1_to_20 & low_ctr
df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = df["reason_code"].map(action_for)

queue = df.sort_values(
    ["score", "impressions_90d", "clicks_90d", "content_id"],
    ascending=[False, False, False, True]
).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)
queue["confidence_note"] = queue.apply(confidence_for, axis=1)
queue["what_would_make_it_wrong"] = queue.apply(what_would_make_it_wrong, axis=1)

# This is the production-style queue: no trend labels or outcome audit fields.
export_columns = [
    "rank", "content_id", "client_id", "action", "reason_code", "score",
    "confidence_note", "what_would_make_it_wrong", "content_type",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "days_since_last_update", "content_age_days"
]
queue_export = queue[export_columns].copy()
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
queue_export.to_csv(output_dir / "baseline_action_score.csv", index=False)

def precision_at_k(ranked, k):
    return float(ranked.head(k)["observed_decline_outcome"].mean())

metrics = {
    "rows_ranked": int(len(queue)),
    "retrospective_decline_base_rate": float(df["observed_decline_outcome"].mean()),
    "precision_at_10_decline_audit_only": precision_at_k(queue, 10),
    "precision_at_50_decline_audit_only": precision_at_k(queue, 50),
    "precision_at_100_decline_audit_only": precision_at_k(queue, 100),
    "score_distribution": {str(k): int(v) for k, v in queue["score"].value_counts().sort_index().items()},
    "reason_code_counts": {str(k): int(v) for k, v in queue["reason_code"].value_counts().items()},
    "score_inputs": score_inputs,
    "forbidden_inputs_excluded": sorted(forbidden_inputs),
}
with open(output_dir / "ml07_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nRanked {len(queue):,} rows and wrote {output_dir / 'baseline_action_score.csv'}")
print("Retrospective decline audit (not used in score):")
print(f"  base rate: {metrics['retrospective_decline_base_rate']:.1%}")
print(f"  precision@10: {metrics['precision_at_10_decline_audit_only']:.1%}")
print(f"  precision@50: {metrics['precision_at_50_decline_audit_only']:.1%}")
print(f"  precision@100: {metrics['precision_at_100_decline_audit_only']:.1%}")
print("\nReason-code counts:")
print(queue["reason_code"].value_counts().to_string())
print("\nTop 10 queue preview:")
print(queue_export.head(10).to_string(index=False))



Ranked 30,000 rows and wrote work/outputs/baseline_action_score.csv
Retrospective decline audit (not used in score):
  base rate: 54.2%
  precision@10: 80.0%
  precision@50: 44.0%
  precision@100: 35.0%

Reason-code counts:
reason_code
visible_page1_2_low_ctr    12597
low_visibility             11248
visible_only                5556
visible_page1_2              577
stale_visible_low_ctr         12
stale_visible                 10

Top 10 queue preview:
 rank           content_id         client_id                    action             reason_code  score                                                                         confidence_note                                                                 what_would_make_it_wrong    content_type  impressions_90d  clicks_90d  ctr  avg_position  days_since_last_update  content_age_days
    1 content_cf56e2e2e282 client_7f2253d7e2       refresh_and_recheck   stale_visible_low_ctr      6 medium-high: two independent review signals, but outcom

## 3. Top-20 review

The assignment requires a top-10 review. I review the top 20 because the starter skeleton asks for it; the first ten rows are the required review set. Each row has an action, one reason code, a confidence note, and a concrete condition that could make the pick wrong.


In [3]:
review_columns = [
    "rank", "content_id", "action", "reason_code", "score",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "days_since_last_update", "confidence_note", "what_would_make_it_wrong"
]
top20_review = queue.head(20)[review_columns].copy()
assert len(top20_review) == 20
print(top20_review.to_string(index=False))
print("\nRequired top-10 review completed; rows 1-20 are shown for extra scrutiny.")


 rank           content_id                    action             reason_code  score  impressions_90d  clicks_90d  ctr  avg_position  days_since_last_update                                                                         confidence_note                                                                 what_would_make_it_wrong
    1 content_cf56e2e2e282       refresh_and_recheck   stale_visible_low_ctr      6            61678          94 0.15          19.7                     194 medium-high: two independent review signals, but outcome still needs human confirmation       The update date may be missing or the page may already be scheduled for a refresh.
    2 content_0a91db491d14       refresh_and_recheck   stale_visible_low_ctr      6            13299          65 0.49          10.5                     193 medium-high: two independent review signals, but outcome still needs human confirmation       The update date may be missing or the page may already be scheduled for a refresh.
 

## 4. Weak picks + leakage check

A useful review should find at least one weak pick. I flag top-ranked rows with very few clicks or low confidence so a human can challenge them before any edit is made.

The leakage check verifies that the score was built from pre-decision fields only. `trend_direction` is the source of the retrospective decline audit, but it never enters `score_inputs`, the score formula, or the exported queue.


In [4]:
weak_picks = queue.head(20).loc[
    (queue.head(20)["clicks_90d"] < 5)
    | (queue.head(20)["confidence_note"].str.startswith("low")),
    review_columns
]
print("Weak picks to challenge:")
print(weak_picks.to_string(index=False))
assert len(weak_picks) >= 1, "The review should challenge at least one weak pick."

# Structural leakage checks. The outcome is deliberately absent from the score and export.
forbidden = {"trend_direction", "trend_pct", "observed_decline_outcome", "is_declining_label"}
assert not forbidden.intersection(score_inputs)
assert not forbidden.intersection(export_columns)
assert set(export_columns).isdisjoint(forbidden)
print("\nLeakage check: PASS")
print(f"Score inputs: {score_inputs}")
print("Excluded from score/export: trend_direction, trend_pct, observed_decline_outcome, is_declining_label")
print("The decline precision numbers above are retrospective audit metrics only.")


Weak picks to challenge:
 rank           content_id                    action             reason_code  score  impressions_90d  clicks_90d  ctr  avg_position  days_since_last_update                                  confidence_note                                                                 what_would_make_it_wrong
   11 content_c8e9d6ab9013 improve_snippet_or_intent visible_page1_2_low_ctr      5           208678           0  0.0           9.7                     104 medium: one strong signal plus useful visibility The 90-day click count may be too small, or the query mix may not match the page intent.

Leakage check: PASS
Score inputs: ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update']
Excluded from score/export: trend_direction, trend_pct, observed_decline_outcome, is_declining_label
The decline precision numbers above are retrospective audit metrics only.


## Self-check

- [x] Two signal checks are shown with bucket tables and `n` values; verdicts are explicit.
- [x] The rule is plain-language, fixed, and uses one reason code plus an action label per row.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] A required top-10 review is present; the notebook also shows the top 20 requested by the skeleton.
- [x] Every reviewed row includes action, reason code, confidence note, and what would make it wrong.
- [x] At least one weak pick is surfaced for human challenge.
- [x] The score uses no future-window or label-derived columns.
- [x] The notebook is intended to run top to bottom from the repository root.
- [x] No client names, URLs, private queries, or tokens are committed.
- [ ] After this local verification, commit the notebook and submit the public repository URL on the ML-07 card.
